<a href="https://colab.research.google.com/github/NehalShahu/Gen_AI/blob/main/Gen_AI_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets evaluate rouge_score sentencepiece accelerate

import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate

print("GPU:", torch.cuda.is_available())

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00
GPU: True


In [2]:
# ==============================
# DATASET
# ==============================

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Small subset for Google Colab
train_data = dataset["train"].select(range(10000))
val_data = dataset["validation"].select(range(1000))
test_data = dataset["test"].select(range(1000))

print("Train:", len(train_data))
print("Validation:", len(val_data))
print("Test:", len(test_data))


# ==============================
# T5 TRANSFORMER MODEL
# ==============================

model_name = "google-t5/t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print("Using:", device)


# ==============================
# TOKENIZATION
# ==============================

def preprocess(examples):

    inputs = [
        "summarize: " + article
        for article in examples["article"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=128,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


train_tokenized = train_data.map(
    preprocess,
    batched=True,
    remove_columns=train_data.column_names
)

val_tokenized = val_data.map(
    preprocess,
    batched=True,
    remove_columns=val_data.column_names
)

test_tokenized = test_data.map(
    preprocess,
    batched=True,
    remove_columns=test_data.column_names
)


# ==============================
# DATA COLLATOR
# ==============================

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)


# ==============================
# TRAINING
# ==============================

training_args = Seq2SeqTrainingArguments(
    output_dir="./news_summarizer",

    eval_strategy="epoch",

    learning_rate=2e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=2,

    weight_decay=0.01,

    predict_with_generate=True,

    fp16=torch.cuda.is_available(),

    save_total_limit=2,

    logging_steps=100,

    report_to="none"
)


trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,

    processing_class=tokenizer,

    data_collator=data_collator
)


# ==============================
# START TRAINING
# ==============================

trainer.train()


# ==============================
# SAVE MODEL
# ==============================

trainer.save_model("./final_news_summarizer")
tokenizer.save_pretrained("./final_news_summarizer")

print("Training completed!")

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training completed!


In [5]:
# ==============================
# EVALUATION
# ==============================

rouge = evaluate.load("rouge")

results = trainer.predict(test_tokenized)

predictions = results.predictions
labels = results.label_ids

# Replace -100 so tokenizer can decode labels
labels = np.where(
    labels != -100,
    labels,
    tokenizer.pad_token_id
)

# Convert tokens back to text
predicted_summaries = tokenizer.batch_decode(
    predictions,
    skip_special_tokens=True
)

actual_summaries = tokenizer.batch_decode(
    labels,
    skip_special_tokens=True
)

# Calculate ROUGE
scores = rouge.compute(
    predictions=predicted_summaries,
    references=actual_summaries
)

print("\n===== ROUGE SCORES =====")

for metric, score in scores.items():
    print(f"{metric}: {score:.4f}")


===== ROUGE SCORES =====
rouge1: 0.2289
rouge2: 0.0904
rougeL: 0.1896
rougeLsum: 0.1894


In [4]:
# ==============================
# CUSTOM NEWS ARTICLE
# ==============================

article = """
The Indian government has announced a new initiative to
increase renewable energy production across the country.
The program will provide financial support for solar and
wind energy projects and encourage private investment.

Officials said the initiative is expected to create thousands
of new jobs while reducing dependence on fossil fuels.
The government will begin accepting applications for the
program later this year.
"""


# ==============================
# GENERATE SUMMARY
# ==============================

inputs = tokenizer(
    "summarize: " + article,
    return_tensors="pt",
    max_length=512,
    truncation=True
).to(device)


summary_ids = model.generate(
    **inputs,
    max_new_tokens=100,
    min_new_tokens=20,
    num_beams=4,
    length_penalty=2.0,
    early_stopping=True
)


summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)


# ==============================
# DISPLAY RESULT
# ==============================

print("========== ORIGINAL ARTICLE ==========\n")
print(article)

print("\n========== GENERATED SUMMARY ==========\n")
print(summary)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=20) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


========== ORIGINAL ARTICLE ==========


The Indian government has announced a new initiative to
increase renewable energy production across the country.
The program will provide financial support for solar and
wind energy projects and encourage private investment.

Officials said the initiative is expected to create thousands
of new jobs while reducing dependence on fossil fuels.
The government will begin accepting applications for the
program later this year.


========== GENERATED SUMMARY ==========

Indian government announces new initiative to increase renewable energy production. The program will provide financial support for solar and wind energy projects. Officials said the initiative is expected to create thousands of new jobs.
